In [ ]:
# %pip install -q lightningrod-ai pandas tqdm python-dateutil

In [ ]:
import os
import re
import json
import time
import inspect
from pathlib import Path
from datetime import datetime, timedelta

import pandas as pd

from lightningrod import (
    LightningRod,
    BinaryAnswerType,
    QuestionPipeline,
    NewsSeedGenerator,
    ForwardLookingQuestionGenerator,
    WebSearchLabeler,
)

try:
    from lightningrod import NewsContextGenerator
    HAS_NEWS_CONTEXT = True
except Exception:
    HAS_NEWS_CONTEXT = False


os.environ["LIGHTNINGROD_API_KEY"] = "..."
API_KEY = os.getenv("LIGHTNINGROD_API_KEY") or "PUT_YOUR_API_KEY_HERE"
assert API_KEY != "PUT_YOUR_API_KEY_HERE", "Вставь свой API key"

client = LightningRod(api_key=API_KEY)
binary_answer = BinaryAnswerType()

FINAL_JSON = Path.home() / "Downloads" / "culture_dataset_final.json"

#даты
START_DATE = datetime(2024, 1, 1)
END_DATE   = datetime(2026, 1, 1)

MIN_ALLOWED_DATE = pd.Timestamp("2024-01-01")
MAX_ALLOWED_PREDICTION_DATE = pd.Timestamp("2026-01-01")
LABEL_SAFE_CUTOFF = pd.Timestamp("2026-03-31")
LABEL_SAFE_CUTOFF_STR = "2026-03-31"

PILOT_MODE = False

if PILOT_MODE:
    TARGET_TOTAL = 50
    BATCH_SIZE = 20
    TEST_DOMAIN_ONLY = "movies_tv"
else:
    TARGET_TOTAL = 1000
    BATCH_SIZE = 250
    TEST_DOMAIN_ONLY = None

WINDOW_DAYS = 45
SLEEP_BETWEEN_CALLS = 2.0

# лучше без контекст-генератора для стабильности
USE_CONTEXT_GENERATOR = False
NUM_CONTEXT_ARTICLES = 8

print("HAS_NEWS_CONTEXT =", HAS_NEWS_CONTEXT)
print("FINAL_JSON =", FINAL_JSON)
print("PILOT_MODE =", PILOT_MODE)

In [ ]:
DEFAULT_PROMPT_TEXT = f"""
Generate high-quality binary forecasting questions specifically about pop culture and entertainment.

Scope:
- film and TV releases, trailers, casting, renewals, premieres, box office
- streaming charts and platform milestones
- music releases, chart outcomes, tour announcements, certifications
- awards nominations and wins
- YouTube / creator milestones and major video outcomes
- celebrity entertainment legal cases and official announcements
- reality TV finalists / eliminations / winners
- gaming as culture: trailers, reveals, release dates, delays, major launch outcomes
- books / adaptations / major festival or entertainment-event outcomes

Rules:
1. Every question must be a YES/NO question.
2. Every question must be objectively verifiable.
3. Every question must have a clear deadline or event date.
4. Avoid subjective language: popular, successful, iconic, beloved, viral, relevant.
5. Do not generate a question if the seed article already states the resolved answer.
6. Prefer questions resolvable by official announcements, recognized charts, box office data,
   award results, court filings, or reputable entertainment press.
7. Keep the wording crisp and prediction-market-like.
8. Focus only on culture / entertainment, not general politics, weather, or macroeconomics.
9. Avoid repetitive bucket ladders and near-duplicate variants.
"""

COMMON_INSTRUCTIONS = DEFAULT_PROMPT_TEXT.strip()

BAD_EXAMPLES = [
    "Will this movie be successful?",
    "Will fans love the album?",
    "Will the song be iconic?",
    "Will the show be popular?",
    "Will the celebrity stay relevant?",
    "Will people be excited about the trailer?",
    "Will the next MrBeast video get 50-60M views in 7 days?",
    "Will the next MrBeast video get 60-70M views in 7 days?",
    "Will the next MrBeast video get 70-80M views in 7 days?",
    "Will this Netflix show be #4 this week?",
    "Will this Netflix show be #5 this week?",
    "Will X or Y happen by Friday?",
    "Will X happen and will Y win?"
]

CULTURE_DOMAINS = [
    {
        "name": "movies_tv",
        "target": 320,
        "search_query": [
            "movie release trailer streaming netflix hbo disney series season cast director entertainment",
            "box office film premiere trailer release date television renewal entertainment"
        ],
        "examples": [
            "Will Netflix release the first trailer for Wednesday Season 2 by 2025-03-31?",
            "Will HBO officially renew House of the Dragon for another season by 2025-06-30?",
            "Will Disney announce the release date for Frozen 3 by 2025-12-31?",
            "Will Superman gross more than $100M in its US opening weekend?"
        ],
    },
    {
        "name": "music",
        "target": 260,
        "search_query": [
            "music album single release billboard grammy tour concert entertainment",
            "artist album release chart top 10 spotify youtube music video entertainment"
        ],
        "examples": [
            "Will Taylor Swift announce a new tour by 2025-08-31?",
            "Will the next Bad Bunny album be released by 2025-10-01?",
            "Will the new Drake single enter the Billboard Hot 100 top 10 within 14 days of release?",
            "Will Sabrina Carpenter have a #1 song on Spotify US by 2025-06-30?"
        ],
    },
    {
        "name": "awards",
        "target": 160,
        "search_query": [
            "oscars emmys grammys bafta golden globes nomination winner entertainment",
            "award season nominee best picture best actor best album entertainment"
        ],
        "examples": [
            "Will Dune: Part Two win Best Picture at the 2025 BAFTAs?",
            "Will Billie Eilish receive a Grammy nomination for Album of the Year in 2025?",
            "Will Shogun receive an Emmy nomination for Outstanding Drama Series in 2025?"
        ],
    },
    {
        "name": "gaming_culture",
        "target": 180,
        "search_query": [
            "video game trailer release delay gameplay reveal entertainment gta vi nintendo playstation",
            "game launch release date delay trailer gaming entertainment"
        ],
        "examples": [
            "Will Rockstar release a new GTA VI trailer by 2025-09-01?",
            "Will Grand Theft Auto VI be delayed beyond 2025-10-31?",
            "Will Nintendo reveal the release date for Metroid Prime 4 by 2025-07-01?"
        ],
    },
    {
        "name": "youtube_creators",
        "target": 80,
        "search_query": [
            "youtube creator mrbeast video milestone subscribers entertainment",
            "youtube trailer views creator collaboration entertainment"
        ],
        "examples": [
            "Will the next MrBeast main-channel video exceed 50 million views within 7 days of release?",
            "Will MrBeast reach 100 million subscribers on his main YouTube channel by 2025-12-31?"
        ],
    },
]

print(COMMON_INSTRUCTIONS[:1200])

In [ ]:
SUBJECTIVE_WORDS = {
    "popular", "successful", "iconic", "beloved", "viral", "relevant",
    "good", "bad", "amazing", "terrible", "great", "flop",
    "big", "huge", "massive", "famous", "important", "major",
    "well-received", "acclaimed", "overrated", "underrated",
    "exciting", "interesting", "impressive", "disappointing"
}

CULTURE_KEYWORDS = {
    "film", "movie", "box office", "opening weekend", "rotten tomatoes",
    "tv", "series", "show", "season", "episode", "finale",
    "trailer", "teaser", "premiere", "release", "renewal",
    "streaming", "netflix", "hbo", "max", "disney", "disney+", "hulu", "prime video",
    "music", "album", "single", "song", "ep", "mixtape", "tour", "concert",
    "spotify", "billboard", "apple music", "chart", "charts", "hot 100",
    "oscar", "oscars", "emmy", "emmys", "grammy", "grammys",
    "bafta", "golden globe", "golden globes", "sag awards",
    "critics choice", "goodreads choice", "award", "awards", "nomination", "winner",
    "youtube", "creator", "mrbeast", "video", "views", "subscribers",
    "podcast", "rogan", "all-in",
    "celebrity", "celeb", "lawsuit", "court", "trial", "divorce",
    "reality", "contestant", "finalist",
    "game", "gaming", "gta", "nintendo", "playstation", "xbox", "steam",
    "app store", "google trends", "trend", "search interest"
}

def daterange_chunks(start_dt: datetime, end_dt: datetime, days: int):
    cur = start_dt
    while cur < end_dt:
        nxt = min(cur + timedelta(days=days), end_dt)
        yield cur, nxt
        cur = nxt

def has_deadline_like_text(question: str) -> bool:
    q = question.lower()
    patterns = [
        r"\bby\b",
        r"\bbefore\b",
        r"\bon\b",
        r"\bwithin\b",
        r"\b\d{4}-\d{2}-\d{2}\b",
        r"\b(january|february|march|april|may|june|july|august|september|october|november|december)\b",
    ]
    return any(re.search(p, q) for p in patterns)

def looks_cultural(question: str) -> bool:
    q = question.lower()
    return any(k in q for k in CULTURE_KEYWORDS)

def passes_quality_filter(question: str) -> bool:
    q = (question or "").strip().lower()
    if len(q) < 20:
        return False
    if not q.endswith("?"):
        return False
    if not (
        q.startswith("will ")
        or q.startswith("is ")
        or q.startswith("are ")
        or q.startswith("does ")
        or q.startswith("did ")
        or q.startswith("can ")
        or q.startswith("has ")
    ):
        return False
    if any(w in q for w in SUBJECTIVE_WORDS):
        return False
    if not looks_cultural(q):
        return False
    if not has_deadline_like_text(q):
        return False
    return True

def dedupe_key(question: str, prediction_date) -> str:
    q = re.sub(r"\s+", " ", (question or "").strip().lower())
    pdv = str(prediction_date)
    return f"{q}|||{pdv}"

def maybe_to_records(flat_obj):
    if isinstance(flat_obj, list):
        return flat_obj
    if isinstance(flat_obj, dict):
        if flat_obj and all(isinstance(v, list) for v in flat_obj.values()):
            n = max(len(v) for v in flat_obj.values())
            return [{k: (v[i] if i < len(v) else None) for k, v in flat_obj.items()} for i in range(n)]
        return [flat_obj]
    if hasattr(flat_obj, "to_dict"):
        try:
            return flat_obj.to_dict(orient="records")
        except Exception:
            pass
    raise TypeError(f"Не удалось превратить flattened() в records: {type(flat_obj)}")

def first_not_none(rec, keys):
    for k in keys:
        if k in rec and rec[k] is not None:
            return rec[k]
    return None

#настройка результата
def to_yes_no(value):
    if value is None:
        return None
    if isinstance(value, bool):
        return "YES" if value else "NO"
    if isinstance(value, (int, float)):
        if value == 1:
            return "YES"
        if value == 0:
            return "NO"
    s = str(value).strip().lower()
    if s in {"1", "true", "yes", "y"}:
        return "YES"
    if s in {"0", "false", "no", "n"}:
        return "NO"
    return None

def to_iso_or_none(value):
    if value is None:
        return None
    try:
        ts = pd.to_datetime(value)
        if pd.isna(ts):
            return None
        return ts.isoformat()
    except Exception:
        return str(value)

def is_prediction_date_ok(value) -> bool:
    if value is None:
        return False
    try:
        ts = pd.to_datetime(value)
        if pd.isna(ts):
            return False
        return MIN_ALLOWED_DATE <= ts <= MAX_ALLOWED_PREDICTION_DATE
    except Exception:
        return False

def infer_question_type(question: str) -> str:
    q = (question or "").lower()
    if any(x in q for x in ["trailer", "teaser", "gameplay showcase"]):
        return "trailer_or_preview"
    if any(x in q for x in ["release", "premiere", "launch", "be released"]):
        return "release_by_date"
    if any(x in q for x in ["announce", "confirm", "reveal"]):
        return "announcement_by_date"
    if any(x in q for x in ["nomination", "nominate", "oscar", "grammy", "emmy", "bafta", "golden globe"]):
        return "awards"
    if any(x in q for x in ["gross more than", "views", "subscribers", "top 10", "billboard", "box office"]):
        return "threshold_milestone"
    if any(x in q for x in ["lawsuit", "court", "settled", "verdict", "dismiss", "trial", "divorce"]):
        return "legal_resolution"
    if any(x in q for x in ["eliminated", "final", "winner", "contestant", "finalist"]):
        return "reality_tv_outcome"
    if any(x in q for x in ["delayed", "delay"]):
        return "delay"
    return "other" 

In [ ]:
def normalize_record(rec: dict, domain_name: str) -> dict:
    question = first_not_none(rec, ["question_text", "question", "text"]) or ""
    prediction_date = first_not_none(rec, ["prediction_date", "forecast_date"])
    resolution_date = first_not_none(rec, ["resolution_date", "resolved_at", "deadline"])

    raw_answer = first_not_none(
        rec,
        ["correct_answer", "answer", "label", "resolved_answer", "ground_truth"]
    )
    reasoning = first_not_none(
        rec,
        ["resolution_reasoning", "reasoning", "label_reasoning", "resolver_reasoning"]
    )

    raw_inner = rec.get("raw_record", {})
    if raw_answer is None and isinstance(raw_inner, dict):
        raw_answer = first_not_none(
            raw_inner,
            ["correct_answer", "answer", "label", "resolved_answer", "ground_truth"]
        )
    if reasoning is None and isinstance(raw_inner, dict):
        reasoning = first_not_none(
            raw_inner,
            ["resolution_reasoning", "reasoning", "label_reasoning", "resolver_reasoning"]
        )
    if resolution_date is None and isinstance(raw_inner, dict):
        resolution_date = first_not_none(
            raw_inner,
            ["resolution_date", "resolved_at", "deadline"]
        )

    answer = to_yes_no(raw_answer)

    prompt = rec.get("prompt") or []
    context_messages = []
    if isinstance(prompt, list):
        for msg in prompt:
            if isinstance(msg, dict):
                role = msg.get("role", "")
                content = msg.get("content", "")
                if content:
                    context_messages.append({"role": role, "content": content})

    return {
        "id": rec.get("sample_id") or rec.get("id"),
        "domain": "culture",
        "subcategory": domain_name,
        "question": question,
        "prediction_date": to_iso_or_none(prediction_date),
        "resolution_date": to_iso_or_none(resolution_date),
        "answer": answer,
        "resolution_reasoning": reasoning,
        "prompt": context_messages,
        "raw_record": rec,
    }

def build_pipeline(domain_cfg: dict, window_start: datetime, window_end: datetime):
    gen_kwargs = {
        "instructions": COMMON_INSTRUCTIONS,
        "examples": domain_cfg["examples"],
    }

    gen_sig = inspect.signature(ForwardLookingQuestionGenerator)
    if "bad_examples" in gen_sig.parameters:
        gen_kwargs["bad_examples"] = BAD_EXAMPLES
    if "questions_per_seed" in gen_sig.parameters:
        gen_kwargs["questions_per_seed"] = 1

    question_generator = ForwardLookingQuestionGenerator(**gen_kwargs)
    pipeline_kwargs = {}

    if HAS_NEWS_CONTEXT and USE_CONTEXT_GENERATOR:
        pipe_sig = inspect.signature(QuestionPipeline)
        if "context_generators" in pipe_sig.parameters:
            pipeline_kwargs["context_generators"] = [NewsContextGenerator(num_articles=NUM_CONTEXT_ARTICLES)]

    pipeline = QuestionPipeline(
        seed_generator=NewsSeedGenerator(
            start_date=window_start,
            end_date=window_end,
            search_query=domain_cfg["search_query"],
        ),
        question_generator=question_generator,
        labeler=WebSearchLabeler(answer_type=binary_answer),
        **pipeline_kwargs
    )
    return pipeline 

In [ ]:
# тестовый прогон на маленьком окне по одному направлению

test_domain = [d for d in CULTURE_DOMAINS if d["name"] == (TEST_DOMAIN_ONLY or "movies_tv")][0]
test_start = datetime(2024, 1, 1)
test_end   = datetime(2024, 9, 30)

pipeline = build_pipeline(test_domain, test_start, test_end)

try:
    try:
        dataset = client.transforms.run(config=pipeline, max_questions=20)
    except TypeError:
        dataset = client.transforms.run(pipeline, max_questions=20)

    flat = dataset.flattened()
    test_records = maybe_to_records(flat)
    test_norm = [normalize_record(r, test_domain["name"]) for r in test_records]

    df_test = pd.DataFrame([
        {
            "question": x["question"],
            "prediction_date": x["prediction_date"],
            "resolution_date": x["resolution_date"],
            "answer": x["answer"],
        }
        for x in test_norm
    ])
    display(df_test.head(20))
    print(df_test["answer"].value_counts(dropna=False))
except Exception as e:
    print("Ошибка на тестовом запуске:", e) 

In [ ]:
def run_generation():
    seen = set()
    final_rows = []
    total_clean = 0

    for domain_cfg in CULTURE_DOMAINS:
        domain_name = domain_cfg["name"]
        domain_target = domain_cfg["target"]
        domain_clean = 0

        if PILOT_MODE and TEST_DOMAIN_ONLY is not None and domain_name != TEST_DOMAIN_ONLY:
            continue

        print(f"\n===== DOMAIN: {domain_name} | target={domain_target} =====")

        for window_start, window_end in daterange_chunks(START_DATE, END_DATE, WINDOW_DAYS):
            if domain_clean >= domain_target or total_clean >= TARGET_TOTAL:
                break

            need_now = min(BATCH_SIZE, domain_target - domain_clean, TARGET_TOTAL - total_clean)
            if need_now <= 0:
                break

            print(f"[{domain_name}] {window_start.date()} -> {window_end.date()} | max_questions={need_now}")

            pipeline = build_pipeline(domain_cfg, window_start, window_end)

            try:
                try:
                    dataset = client.transforms.run(config=pipeline, max_questions=need_now)
                except TypeError:
                    dataset = client.transforms.run(pipeline, max_questions=need_now)

                flat = dataset.flattened()
                records = maybe_to_records(flat)
            except Exception as e:
                print("Ошибка:", e)
                time.sleep(SLEEP_BETWEEN_CALLS)
                continue

            added_now = 0

            for rec in records:
                norm = normalize_record(rec, domain_name)
                q = norm.get("question") or ""
                key = dedupe_key(q, norm.get("prediction_date"))

                if not passes_quality_filter(q):
                    continue
                if key in seen:
                    continue
                if norm.get("answer") not in {"YES", "NO"}:
                    continue
                if not norm.get("resolution_date"):
                    continue
                if not is_prediction_date_ok(norm.get("prediction_date")):
                    continue

                seen.add(key)

                final_row = {
                    "id": norm.get("id"),
                    "domain": norm.get("domain"),
                    "subcategory": norm.get("subcategory"),
                    "question_type": infer_question_type(norm.get("question", "")),
                    "question": norm.get("question"),
                    "prediction_date": norm.get("prediction_date"),
                    "resolution_date": norm.get("resolution_date"),
                    "answer": norm.get("answer"),
                    "resolution_reasoning": norm.get("resolution_reasoning"),
                    "prompt": norm.get("prompt"),
                }

                final_rows.append(final_row)
                added_now += 1
                domain_clean += 1
                total_clean += 1

            print(
                f"added={added_now:4d} | "
                f"domain_total={domain_clean:4d} | "
                f"global_total={total_clean:4d}"
            )

            time.sleep(SLEEP_BETWEEN_CALLS)

        print(f"---- done {domain_name}: {domain_clean} clean questions ----")

        if total_clean >= TARGET_TOTAL:
            print("\nДошли до TARGET_TOTAL.")
            break

    return final_rows

In [ ]:
# большая генерация
final_rows = run_generation()

with FINAL_JSON.open("w", encoding="utf-8") as f:
    json.dump(final_rows, f, ensure_ascii=False, indent=2)

print("Готово.")
print("Файл сохранён сюда:")
print(FINAL_JSON)
print("Количество строк:", len(final_rows))

if final_rows:
    df_final = pd.DataFrame(final_rows)
    print(df_final["answer"].value_counts(dropna=False))
    display(df_final.head(20))

In [ ]:
# Открыть папку с финальным файлом
import os
os.startfile(FINAL_JSON.parent)